# Multi-seed sweep and silencing analysis

This notebook combines two kinds of generated artifacts:

- **Sweep collector manifests** (`sweep_archive_*/manifest.json`) for conventional selected scores.
- **Silencing counterfactual directories** (`silencing_*`) for intervention statistics and figures.

Both inputs accept lists, so they may be one combined two-seed directory or separate directories for each seed. The configuration cell also supports including/excluding configurations and seeds.

The notebook is descriptive. Individual seed points remain visible; it does not treat recording interventions as independent samples for significance testing.


In [ ]:
from pathlib import Path

# Locate baselines/dcase2024_task5 whether Jupyter starts there or in notebooks/.
CWD = Path.cwd().resolve()
if (CWD / "tools" / "multiseed_results.py").is_file():
    TASK_ROOT = CWD
elif (CWD.parent / "tools" / "multiseed_results.py").is_file():
    TASK_ROOT = CWD.parent
else:
    raise RuntimeError("Start Jupyter in dcase2024_task5 or its notebooks directory")

# Use one combined archive/analysis directory, or list separate seed directories.
SWEEP_MANIFESTS = [
    TASK_ROOT / "analysis" / "sweep_archive_seeds1234_2026" / "manifest.json",
    TASK_ROOT / "analysis" / "sweep_archive_seed_3" / "manifest.json",
]
SILENCING_DIRS = [
    TASK_ROOT / "analysis" / "silencing_seeds1234_2026",
    TASK_ROOT / "analysis" / "silencing_seed3009",
]

# Configuration keys from the runner. None means include all.
INCLUDE_CONFIGURATIONS = None
# Example: ["baseline", "phase1_random", "phase2_energy"]
EXCLUDE_CONFIGURATIONS = []

# None means include all seeds found in the input files.
INCLUDE_SEEDS = None
# Example: [1234, 2025]

PRIMARY_EVALUATOR = "corrected"  # "corrected" or "released"
TOP_RECORDINGS = 12

OUTPUT_DIR = TASK_ROOT / "analysis" / "paper_outputs_three_seeds"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Task root:", TASK_ROOT)
print("Output:", OUTPUT_DIR)


In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(TASK_ROOT / "tools"))
from multiseed_results import (
    DISPLAY_NAMES,
    configuration_order,
    conventional_summary,
    evaluator_summary,
    filter_all,
    intervention_prevalence,
    load_silencing_directories,
    load_sweep_manifests,
    paired_deltas,
    paired_delta_summary,
    recording_recurrence,
    stratified_intervention_summary,
    validate_results,
)

sns.set_theme(style="whitegrid", context="paper")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

scores_all = load_sweep_manifests(SWEEP_MANIFESTS)
silencing_all = load_silencing_directories(SILENCING_DIRS)

scores, silencing = filter_all(
    scores_all,
    silencing_all,
    include_configurations=INCLUDE_CONFIGURATIONS,
    exclude_configurations=EXCLUDE_CONFIGURATIONS,
    include_seeds=INCLUDE_SEEDS,
)

validation = validate_results(scores, silencing)
print(validation)
print("Configurations:", configuration_order(scores))
print("Seeds:", sorted(scores["seed"].unique()))


## Conventional sweep performance

The first table shows every selected run. The summary is descriptive: mean, sample SD, and range across the included seeds.


In [ ]:
score_columns = [
    "configuration_label",
    "configuration",
    "seed",
    "fmeasure_harmonic",
    "fmeasure_arithmetic",
    "precision_harmonic",
    "recall_harmonic",
    "detection_threshold",
    "postprocessing_type",
    "postprocessing_threshold",
]

display(
    scores[score_columns]
    .sort_values(["seed", "fmeasure_harmonic"], ascending=[True, False])
    .reset_index(drop=True)
)

conventional = conventional_summary(scores)
display(conventional)
conventional.to_csv(TABLE_DIR / "conventional_multiseed_summary.csv", index=False)


## Paired differences from the same-seed baseline

Keep `baseline` in the included configurations for this analysis. Positive values mean that the extension beats the baseline for that seed.


In [ ]:
delta = paired_deltas(scores)
delta = delta.reindex(columns=configuration_order(scores, include_baseline=False))
delta_summary = paired_delta_summary(delta)

display(delta)
display(delta_summary)

delta.to_csv(TABLE_DIR / "paired_deltas_by_seed.csv")
delta_summary.to_csv(TABLE_DIR / "paired_delta_summary.csv")


In [ ]:
configurations = list(delta.columns)
x = np.arange(len(configurations))

fig, ax = plt.subplots(figsize=(7.2, 3.5))
for seed, row in delta.iterrows():
    ax.plot(
        x,
        row[configurations],
        marker="o",
        linewidth=1.2,
        markersize=4.5,
        alpha=0.85,
        label="Seed %s" % seed,
    )

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY_NAMES.get(c, c) for c in configurations], rotation=32, ha="right")
ax.set_ylabel(r"Paired $\Delta F_{\mathrm{harm}}$ vs. baseline")
ax.set_xlabel("")
ax.legend(frameon=False, ncol=max(1, len(delta.index)))
fig.tight_layout()
fig.savefig(FIGURE_DIR / "paired_seed_deltas.pdf", bbox_inches="tight")
fig.savefig(FIGURE_DIR / "paired_seed_deltas.png", dpi=220, bbox_inches="tight")
plt.show()


## Silencing prevalence

Files already absent from a prediction CSV are no-op controls and are excluded from the eligible-intervention denominator.


In [ ]:
interventions = silencing["interventions"]
subset_effects = silencing["subset_effects"]

assert subset_effects["condition_matches_observation"].all(), (
    "Analytical condition mismatches found",
    subset_effects.loc[~subset_effects["condition_matches_observation"]],
)

by_evaluator = evaluator_summary(interventions)
by_target_status = stratified_intervention_summary(interventions)
by_run = intervention_prevalence(interventions)

display(by_evaluator)
display(by_target_status)
display(
    by_run.sort_values(
        ["evaluator_variant", "seed", "aggregate_improvements"],
        ascending=[True, True, False],
    )
)

by_evaluator.to_csv(TABLE_DIR / "silencing_by_evaluator.csv", index=False)
by_target_status.to_csv(
    TABLE_DIR / "silencing_stratified_by_target_tp.csv", index=False
)
by_run.to_csv(TABLE_DIR / "silencing_by_run.csv", index=False)


## Top interventions under the primary evaluator

Prefer a representative file that recurs across seeds/configurations over the single largest post-hoc gain.


In [ ]:
eligible = interventions[~interventions["originally_silent"]].copy()
primary = eligible[eligible["evaluator_variant"] == PRIMARY_EVALUATOR].copy()

top_interventions = primary.sort_values(
    "delta_fmeasure_harmonic", ascending=False
)[[
    "configuration_label",
    "configuration",
    "seed",
    "target_filename",
    "affected_subsets",
    "predicted_events_removed",
    "predicted_duration_removed_seconds",
    "max_delta_affected_subset_f",
    "delta_fmeasure_harmonic",
    "delta_precision_harmonic",
    "delta_recall_harmonic",
    "improves_official",
]]

display(top_interventions.head(30))
top_interventions.head(50).to_csv(
    TABLE_DIR / ("top50_%s_interventions.csv" % PRIMARY_EVALUATOR),
    index=False,
)


In [ ]:
ranked = primary.sort_values(
    "delta_fmeasure_harmonic", ascending=False
).reset_index(drop=True)
ranked["rank"] = np.arange(1, len(ranked) + 1)
ranked["subset"] = ranked["affected_subsets"].astype(str).str.split(";").str[0]

fig, ax = plt.subplots(figsize=(7.2, 3.6))
sns.scatterplot(
    data=ranked,
    x="rank",
    y="delta_fmeasure_harmonic",
    hue="subset",
    style="seed",
    s=28,
    alpha=0.78,
    linewidth=0,
    ax=ax,
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Single-recording interventions ranked by aggregate change")
ax.set_ylabel(r"$\Delta F_{\mathrm{harm}}$")
ax.legend(title="Subset / seed", frameon=False, ncol=3, fontsize=7)
fig.tight_layout()
fig.savefig(FIGURE_DIR / ("ranked_%s_silencing_deltas.pdf" % PRIMARY_EVALUATOR), bbox_inches="tight")
fig.savefig(FIGURE_DIR / ("ranked_%s_silencing_deltas.png" % PRIMARY_EVALUATOR), dpi=220, bbox_inches="tight")
plt.show()


## Recording recurrence and heatmap

This identifies recordings whose aggregate-improving silence effect recurs across independently trained seeds or configurations.


In [ ]:
recurrence = recording_recurrence(
    interventions, PRIMARY_EVALUATOR, require_true_positive=True
)
display(recurrence.head(30))
recurrence.to_csv(
    TABLE_DIR / ("recording_recurrence_%s_tpos.csv" % PRIMARY_EVALUATOR)
)

universal = recurrence[
    recurrence["n_aggregate_improvements"] == recurrence["n_interventions"]
].sort_values(["mean_delta", "max_delta"], ascending=False)
top_files = list(universal.head(TOP_RECORDINGS).index)
heatmap_source = primary[
    primary["target_filename"].isin(top_files) & (primary["target_tp"] > 0)
].copy()

config_order = [
    "baseline", "cife", "phase1_random", "slot_uniform",
    "phase2_energy", "phase3_activation", "afri_direct", "afri_inverse",
]
config_labels = ["Base", "CIFE", "Random", "Slot", "Energy", "Activ.", "AFRI-D", "AFRI-I"]
seed_values = sorted(heatmap_source["seed"].unique())
seed_labels = {seed: "Seed %s" % chr(ord("A") + index) for index, seed in enumerate(seed_values)}
heatmap_source["column_key"] = heatmap_source.apply(
    lambda row: "%s|%s" % (int(row["seed"]), row["configuration"]), axis=1
)
column_order = [
    "%s|%s" % (seed, configuration)
    for seed in seed_values
    for configuration in config_order
]
heatmap_data = heatmap_source.pivot_table(
    index="target_filename", columns="column_key",
    values="delta_fmeasure_harmonic", aggfunc="first",
).reindex(index=top_files, columns=column_order)

fig, ax = plt.subplots(figsize=(3.5, 3.25))
ax.set_facecolor("#d9d9d9")
sns.heatmap(
    heatmap_data, mask=heatmap_data.isna(), cmap="YlOrRd", vmin=0,
    linewidths=0.18, linecolor="white",
    cbar_kws={"label": r"$\Delta F_{\mathrm{harm}}$", "shrink": 0.70, "pad": 0.02},
    ax=ax,
)
ax.set_xticks([position + 0.5 for position in range(len(column_order))])
ax.set_xticklabels(
    config_labels * len(seed_values), rotation=62, ha="right",
    rotation_mode="anchor", fontsize=5.5,
)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=6)
ax.set_xlabel("Configuration within training replicate", fontsize=7)
ax.set_ylabel("")
ax.tick_params(axis="both", length=0)
for boundary in range(1, len(seed_values)):
    ax.axvline(boundary * len(config_order), color="black", linewidth=1.0)
for index, seed in enumerate(seed_values):
    center = (index * len(config_order) + len(config_order) / 2) / len(column_order)
    ax.text(
        center, 1.035, seed_labels[seed], transform=ax.transAxes,
        ha="center", va="bottom", fontsize=7, fontweight="bold",
    )
fig.tight_layout(pad=0.35, rect=(0, 0, 1, 0.97))
fig.savefig(
    FIGURE_DIR / ("recurring_file_heatmap_%s_tpos.pdf" % PRIMARY_EVALUATOR),
    bbox_inches="tight",
)
fig.savefig(
    FIGURE_DIR / ("recurring_file_heatmap_%s_tpos.png" % PRIMARY_EVALUATOR),
    dpi=350, bbox_inches="tight",
)
plt.show()


## Released versus corrected evaluator

Each point is the same configuration, seed, and target recording under the two evaluator variants.


In [ ]:
variant_comparison = interventions.pivot_table(
    index=["configuration", "configuration_label", "seed", "target_filename"],
    columns="evaluator_variant",
    values="delta_fmeasure_harmonic",
    aggfunc="first",
).reset_index()

if {"released", "corrected"}.issubset(variant_comparison.columns):
    fig, ax = plt.subplots(figsize=(4.4, 4.1))
    sns.scatterplot(
        data=variant_comparison,
        x="released",
        y="corrected",
        hue="seed",
        alpha=0.68,
        s=28,
        linewidth=0,
        ax=ax,
    )
    low = min(variant_comparison["released"].min(), variant_comparison["corrected"].min())
    high = max(variant_comparison["released"].max(), variant_comparison["corrected"].max())
    ax.plot([low, high], [low, high], "--", color="gray", linewidth=0.8)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.axvline(0, color="black", linewidth=0.6)
    ax.set_xlabel(r"Released evaluator $\Delta F_{\mathrm{harm}}$")
    ax.set_ylabel(r"Corrected evaluator $\Delta F_{\mathrm{harm}}$")
    ax.legend(title="Seed", frameon=False)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "released_vs_corrected.pdf", bbox_inches="tight")
    fig.savefig(FIGURE_DIR / "released_vs_corrected.png", dpi=220, bbox_inches="tight")
    plt.show()
else:
    print("Both evaluator variants are required for this plot.")


## Compact tables for paper drafting

These exports are intentionally plain. Format the final LaTeX tables manually for IEEE column width after deciding which columns survive the page limit.


In [ ]:
paper_silence_summary = by_evaluator.copy()
paper_silence_summary["subset_improvement_pct"] = (
    100 * paper_silence_summary["subset_improvements"]
    / paper_silence_summary["eligible_interventions"]
)
paper_silence_summary["aggregate_improvement_pct"] = (
    100 * paper_silence_summary["aggregate_improvements"]
    / paper_silence_summary["eligible_interventions"]
)

display(paper_silence_summary)

paper_silence_summary.to_csv(
    TABLE_DIR / "paper_silencing_summary.csv", index=False
)
paper_silence_summary.to_latex(
    TABLE_DIR / "paper_silencing_summary.tex",
    index=False,
    float_format="%.3f",
)

conventional.to_latex(
    TABLE_DIR / "paper_conventional_summary.tex",
    index=False,
    float_format="%.3f",
)

print("Figures written to:", FIGURE_DIR)
print("Tables written to:", TABLE_DIR)
